## GoTravel AI Travel Assistant Agent (RAG System)

GoTravel is an online travel agency that helps customers plan and book flights, hotels and holiday packages. The agency's goal us to make travel planning simple, fast and reliable. however, GoTravel's customer support team struggles to provide timelt and accurate answers to customer queries.\
Customers often wait too long for answers to policy-related questions, and even then, the information is sometimes incomplete or inconsitent. This leads to repeated inquiries, high support costs, and growing customer dissatisfaction. If unresolved, these issues will negatively impact GoTravel's reputation and customer loyalty.

For GoTravel's RAG System, the following libraries are required:
* sentence-transformers: provides pretrained models for generating vectors for sentences
* faiss-cpu: performs efficient similarity search and clustering of dense vectors
* langchain_community: provides intergrations with various tools and services, including LLM's
* replicate: Enables inyteration with models hosted on the Replicate platform

In [ ]:
!pip install sentence-transformers faiss-cpu langchain_community replicate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from langchain_community.llms import Replicate
import os
from google.colab import userdata

/tmp/ipykernel_553/609613177.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Replicate


### Task 2: Configure and test the LLM for baseline repsonses

In this task, you'll connect to the Replicate application programming interface(API) and set up the IMB Granite language model, which will serve as the LLM powering GoTravel's travel assistant. You'll then run a baseline user query to review how the LLM responds using only its built-in knowledge.\
A baseline query is an initial question you ask the LLM without providing any external documents or context. The LLM's reply to this query is called the baseline response which the LLM genrated using its built-in knowledge.\
You are running a baseline query to assess how the LLM answers customer questions about travel without referring to the Gotravel's internal policies or other important information. This will give you a basis for comparison after enhancing the LLM's knowledge base.

In [ ]:
api_token = userdata.get('Replicate_api_token')
os.environ["REPLICATE_API_TOKEN"] = api_token

model = "anthropic/claude-4.5-sonnet"
output = Replicate(
    model=model,
    replicate_api_token=api_token)

Once the API token is successfully loaded, you can test the Replicate API connection by making a simple call to the model. We'll use the `invoke` method of the `Replicate` object to send a test query.

In [ ]:
user_query = input ("Ask a question: ")
initial_response = output.invoke(user_query)
print("\nInitial Response (No RAG):\n")
print(initial_response)

Ask a question: I'm a citizen of Romania. Do I need a schegen visa to travel to France?

Initial Response (No RAG):

No, you do **not** need a Schengen visa to travel to France.

Romania is an EU member state, and as a Romanian citizen, you have the right to free movement within the European Union. You can travel to, live, and work in France (and other EU countries) with just your valid Romanian passport or national ID card.

However, please note:
- **For short visits**: Your Romanian ID card or passport is sufficient
- **For stays longer than 3 months**: You may need to register with local authorities in France

While Romania is not yet part of the Schengen Area for land and sea borders (as of my last update), this doesn't affect your right as an EU citizen to travel visa-free to France and other EU/Schengen countries.

Have a great trip!


### Task 3: Set up a RAG system for context-based answering

Chunking splits knowledge sources into smalled pieces called chunks. Each chunk slightly overlaps with the next to keep context. This makes it easier for LLMs to search and retreive relevant information.


In [ ]:
from google.colab import files
uploaded = files.upload()
doc_path = list(uploaded.keys())[0]

with open(doc_path, 'r', encoding='utf-8') as f:
   document_text = f.read()

def split_into_chunks(text, chunk_size=300, overlap=50):
   chunks = []
   start = 0
   while start < len(text):
       end = start + chunk_size
       chunks.append(text[start:end])
       start += chunk_size - overlap
   return chunks
chunks = split_into_chunks(document_text)
print(f"Document split into {len(chunks)} chunks.")


Saving External_KB_for_RAG.txt to External_KB_for_RAG.txt
Document split into 10 chunks.


In this step, you prepare the chunks for effecient retreival by converting them into embeddings and storing them in a FAISS index. The code includes the following sections:\
* **Initializes the embedding model:** Loads thr pretrained SentenceTransformer model(all-MiniLM-L6-v2) to convert text xhunks into vector embeddings.\
* **Generates embeddings:** Creates embeddings for each chunk and stores them\
* **Determines embedding dimensions**: Caluclates the vector size required to create a FAISS index\
* **Creates a FAISS index:** Initializes an IndexFlatL2 index, an index type that uses a mathematical technique to perform similarity search.\
* **Adds embeddings to the index:** Populates the FAISS index with the generated embeddings, making them searchable\
* **Stores original chunks:** Saves the text chunks separately in stores chunks because FAISS only stores numerical data, not the original text\
* **Prints a confirmation:** Displays how many embeddings were stored, confirming the setup worked

In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embed_model.encode(chunks)
dimension = embeddings.shape[1]

# Initialize FAISS index
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# Keep chunks for later retrieval
stored_chunks = chunks

print(f"Stored {len(stored_chunks)} embeddings in FAISS index.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Stored 10 embeddings in FAISS index.


In [ ]:
query_embedding = embed_model.encode([user_query])
D, I = index.search(query_embedding, k=3)

top_chunks = [stored_chunks[i] for i in I[0]]
context = "\n".join(top_chunks)

rag_prompt = f"""Use the following context to answer the question.

Context:
{context}

Question:
{user_query}
"""

rag_response = output.invoke(rag_prompt)


### Task 4: Compare baseline responses with contect-augmented answers

In this task, you'll compare baseline and RAG responses to assess how real-time access to GoTravel's internal documents imporves the accuracy and relevance of responses to customer queries

In [ ]:
from IPython.display import display, Markdown

display(Markdown(f"""
# Original Query

**{user_query}**

--------------

# Without RAG

{initial_response}

 ------------------

# With RAG (Context-Augmented)

{rag_response}
"""))


 
# Original Query 

**I'm a citizen of Romania. Do I need a schegen visa to travel to France?** 

-------------- 

# Without RAG 

No, you do **not** need a Schengen visa to travel to France.

Romania is an EU member state, and as a Romanian citizen, you have the right to free movement within the European Union. You can travel to, live, and work in France (and other EU countries) with just your valid Romanian passport or national ID card.

However, please note:
- **For short visits**: Your Romanian ID card or passport is sufficient
- **For stays longer than 3 months**: You may need to register with local authorities in France

While Romania is not yet part of the Schengen Area for land and sea borders (as of my last update), this doesn't affect your right as an EU citizen to travel visa-free to France and other EU/Schengen countries.

Have a great trip!

 ------------------ 

# With RAG (Context-Augmented) 

# Answer

**No, you do not need a Schengen visa to travel to France.**

As a citizen of Romania, you are an EU citizen. Romania is joining the Schengen Area as of **January 1, 2025**.

## Important Information:

- **Current status (before January 1, 2025)**: Based on the context provided, the pre-RAG system response suggested a visa was needed, but this appears to be incorrect as Romania has been an EU member state.

- **After January 1, 2025**: You will definitely not need a Schengen visa. However, you may need to apply for an **ETIAS permit** for short-term stays:
  - Valid for up to 90 days in a 180-day period
  - Valid for up to 3 years or until passport expiration
  - Costs €7 (for applicants aged 18-70)
  - Processed within 3 days of application

Since your query is dated **November 30, 2024** (before January 1, 2025), please verify current requirements with official sources for immediate travel. For travel after January 1, 2025, no Schengen visa will be required, though ETIAS may apply. 
